# 5. ONNX Runtime inference

The cached ONNX reader uses CPUExecutionProvider with NumPy/Pillow preprocessing. It loads neither PyTorch CUDA nor TensorRT. Image lists are processed sequentially by this static-batch-one backend.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
RUN_ONNX = (deployment / "decoder_cached_step.onnx").is_file()
if RUN_ONNX:
    run("infer", "--backend", "onnx", "--image", str(IMAGE_PATH))
    run("evaluate", "--backend", "onnx", "--folder", str(config["dataset_root"] / "val"), "--output", str(config["output_dir"] / "evaluation_onnx"))
else:
    print("Export first:", deployment)


In [ ]:
RUN_NATIVE_PARITY = RUN_ONNX and checkpoint.is_file()
if RUN_NATIVE_PARITY:
    from PIL import Image
    from waterec_ocr.waterec_models import TorchRecognizer
    from waterec_ocr.waterec_onnx import ONNXRecognizer
    native = TorchRecognizer(config, checkpoint)
    onnx_reader = ONNXRecognizer(deployment)
    images = [Image.open(p).convert("RGB") for p in examples]
    try:
        expected, actual = native.infer(images), onnx_reader.infer(images)
        print(list(zip(expected, actual)))
        assert expected == actual, "Prediction parity failed; investigate before deployment"
    finally:
        onnx_reader.close()
